# Step 2: Fair Baseline — Fine-tune `bge-base-en-v1.5`

**Goal:** Train `bge-base-en-v1.5` on the *exact same data splits* as `all-MiniLM-L6-v2`.
The only variable that changes is the base model. This isolates fine-tuning effect from model-size effect.

**Two runs:**
- `bge_arxiv100` — 2700 arXiv-only pairs (mirrors existing arXiv-only MiniLM)
- `bge_mixed50` — 50/50 arXiv+MARCO mix (mirrors existing mixed_50 MiniLM)

**Same config as all previous runs:**
- Loss: `MultipleNegativesRankingLoss`
- Epochs: 5, Batch: 32, LR: 2e-5, Warmup: 100 steps
- Train: first 2700 pairs, Val: pairs 2700–2900, Held-out eval: last 100 (never touched)

**Output:** Two saved models + FAISS indexes, ready to drop into your existing eval scripts.

## 0. Runtime check — confirm T4 GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip())
if 'T4' not in result.stdout:
    print('WARNING: Not on T4. Go to Runtime > Change runtime type > T4 GPU')
else:
    print('Good to go.')

## 1. Install dependencies

In [ ]:
%%capture
!pip install sentence-transformers faiss-cpu datasets accelerate

## 2. Mount Google Drive and set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Auto-find the project folder by searching for training_pairs.json ─────────
def find_project_dir(search_root='/content/drive/MyDrive', marker='training_pairs.json'):
    for root, dirs, files in os.walk(search_root):
        # Skip hidden folders and very deep paths to keep it fast
        dirs[:] = [d for d in dirs if not d.startswith('.')]
        if marker in files:
            # marker is in data/, so project root is one level up
            return os.path.dirname(root)
    return None

print('Searching for project folder in Google Drive...')
PROJECT_DIR = find_project_dir()

if PROJECT_DIR is None:
    raise FileNotFoundError(
        'Could not find training_pairs.json anywhere in MyDrive. '
        'Make sure your project folder is uploaded/synced to Google Drive.'
    )

print(f'Found project at: {PROJECT_DIR}')

DATA_DIR        = os.path.join(PROJECT_DIR, 'data')
EXPERIMENTS_DIR = os.path.join(PROJECT_DIR, 'experiments')

# Verify key files exist
for f in ['training_pairs.json', 'marco_pairs.json', 'manual_eval.json']:
    path = os.path.join(DATA_DIR, f)
    status = '✓' if os.path.exists(path) else '✗ MISSING'
    print(f'{status}  {f}')

## 3. Config — identical to all previous runs

In [ ]:
import random
import numpy as np

BASE_MODEL   = 'BAAI/bge-base-en-v1.5'  # <-- only change vs previous runs
TRAIN_SIZE   = 2700
EPOCHS       = 5
BATCH_SIZE   = 32
WARMUP_STEPS = 100
SEED         = 42   # same seed as all previous runs

random.seed(SEED)
np.random.seed(SEED)

print(f'Base model : {BASE_MODEL}')
print(f'Train size : {TRAIN_SIZE}')
print(f'Epochs     : {EPOCHS}')
print(f'Batch size : {BATCH_SIZE}')

## 4. Load data

In [ ]:
import json

# ── arXiv pairs ───────────────────────────────────────────────────────────────
with open(os.path.join(DATA_DIR, 'training_pairs.json')) as f:
    all_arxiv = json.load(f)

# Exact same split as all previous runs
train_pool  = all_arxiv[:-100]          # 2900 available for training
held_out    = all_arxiv[-100:]          # last 100 — never used in training or val
train_arxiv = train_pool[:TRAIN_SIZE]   # first 2700 = training
val_pairs   = train_pool[TRAIN_SIZE:]   # 2700–2900 = validation (200 pairs)

print(f'arXiv train : {len(train_arxiv)}')
print(f'Validation  : {len(val_pairs)}')
print(f'Held-out    : {len(held_out)}  (untouched)')

# ── MS MARCO pairs ────────────────────────────────────────────────────────────
with open(os.path.join(DATA_DIR, 'marco_pairs.json')) as f:
    all_marco = json.load(f)
print(f'MARCO cache : {len(all_marco)} pairs')

## 5. Build training sets

In [ ]:
def mix_pairs(arxiv_pairs, marco_pairs, arxiv_pct, total=TRAIN_SIZE, seed=SEED):
    rng = random.Random(seed)
    n_arxiv = int(total * arxiv_pct / 100)
    n_marco = total - n_arxiv
    a = rng.sample(arxiv_pairs, min(n_arxiv, len(arxiv_pairs)))
    m = rng.sample(marco_pairs, min(n_marco, len(marco_pairs)))
    mixed = a + m
    rng.shuffle(mixed)
    print(f'  {len(a)} arXiv + {len(m)} MARCO = {len(mixed)} total ({arxiv_pct}/{100-arxiv_pct})')
    return mixed

print('Building training sets...')
print('arXiv-only (100/0):')
set_arxiv100 = mix_pairs(train_arxiv, [], 100)      # all arXiv
print('Mixed 50/50:')
set_mixed50  = mix_pairs(train_arxiv, all_marco, 50)

## 6. Build validation evaluator (same for both runs)

In [ ]:
from sentence_transformers.evaluation import InformationRetrievalEvaluator

def make_evaluator(val_pairs, name):
    queries  = {str(i): p['query']    for i, p in enumerate(val_pairs)}
    corpus   = {str(i): p['positive'] for i, p in enumerate(val_pairs)}
    relevant = {str(i): {str(i)}      for i in range(len(val_pairs))}
    return InformationRetrievalEvaluator(queries, corpus, relevant, name=name)

evaluator = make_evaluator(val_pairs, name='val-bge')
print(f'Validation evaluator ready — {len(val_pairs)} query-passage pairs')

## 7. Training function

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample
from sentence_transformers.losses import MultipleNegativesRankingLoss
from torch.utils.data import DataLoader

def train_bge(pairs, run_name, evaluator):
    out_dir = os.path.join(EXPERIMENTS_DIR, run_name, 'model')
    os.makedirs(out_dir, exist_ok=True)

    examples = [InputExample(texts=[p['query'], p['positive']]) for p in pairs]
    model    = SentenceTransformer(BASE_MODEL)
    loader   = DataLoader(examples, shuffle=True, batch_size=BATCH_SIZE)
    loss     = MultipleNegativesRankingLoss(model)

    print(f'\nTraining {run_name} — {len(pairs)} pairs, {EPOCHS} epochs')
    print(f'Output: {out_dir}')

    model.fit(
        train_objectives=[(loader, loss)],
        evaluator=evaluator,
        epochs=EPOCHS,
        warmup_steps=WARMUP_STEPS,
        evaluation_steps=len(loader),   # eval every epoch
        output_path=out_dir,
        save_best_model=True,
        show_progress_bar=True
    )

    print(f'Saved to {out_dir}')
    return out_dir

## 8a. Train — bge arXiv-only (~10 min on T4)

In [ ]:
arxiv100_dir = train_bge(set_arxiv100, 'bge_arxiv100', evaluator)

## 8b. Train — bge mixed 50/50 (~10 min on T4)

In [ ]:
mixed50_dir = train_bge(set_mixed50, 'bge_mixed50', evaluator)

## 9. Build FAISS indexes for both models

In [ ]:
import pickle
import faiss

# Load corpus chunks from your existing baseline index
# (same chunks used for all other models)
baseline_chunks_path = os.path.join(EXPERIMENTS_DIR, 'baseline', 'baseline_chunks.pkl')
with open(baseline_chunks_path, 'rb') as f:
    chunks = pickle.load(f)
print(f'Loaded {len(chunks)} corpus chunks')

def build_faiss_index(model_dir, run_name):
    print(f'\nBuilding FAISS index for {run_name}...')
    model = SentenceTransformer(model_dir)

    texts = [c['text'] for c in chunks]
    embeddings = model.encode(texts, batch_size=256, show_progress_bar=True,
                               normalize_embeddings=True)

    dim   = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)

    exp_dir   = os.path.join(EXPERIMENTS_DIR, run_name)
    idx_path  = os.path.join(exp_dir, f'{run_name}.faiss')
    faiss.write_index(index, idx_path)
    print(f'Saved index: {idx_path}  ({index.ntotal} vectors, dim={dim})')
    return idx_path

build_faiss_index(arxiv100_dir, 'bge_arxiv100')
build_faiss_index(mixed50_dir,  'bge_mixed50')

## 10. Quick sanity-check eval on held-out synthetic set (n=100)

In [ ]:
def hit_rate_at_k(model_dir, index_path, eval_pairs, k=3):
    model = SentenceTransformer(model_dir)
    index = faiss.read_index(index_path)

    hits = 0
    for p in eval_pairs:
        q_emb = model.encode([p['query']], normalize_embeddings=True)
        _, ids = index.search(q_emb, k)
        retrieved_texts = [chunks[i]['text'] for i in ids[0] if i < len(chunks)]
        # Semantic match: check if positive passage is in any retrieved chunk
        pos = p['positive']
        if any(pos[:100] in r for r in retrieved_texts):
            hits += 1

    hr = hits / len(eval_pairs)
    return hr

print('Sanity check — Hit@3 on held-out synthetic (n=100):')
for run in ['bge_arxiv100', 'bge_mixed50']:
    model_dir  = os.path.join(EXPERIMENTS_DIR, run, 'model')
    index_path = os.path.join(EXPERIMENTS_DIR, run, f'{run}.faiss')
    hr = hit_rate_at_k(model_dir, index_path, held_out)
    print(f'  {run:20s}  Hit@3 = {hr:.4f}')

## 11. Done — what to do next

Both models and indexes are saved to your Drive. Back on your M2:

```
experiments/
  bge_arxiv100/
    model/          ← fine-tuned bge-base, arXiv-only
    bge_arxiv100.faiss
  bge_mixed50/
    model/          ← fine-tuned bge-base, 50/50 mix
    bge_mixed50.faiss
```

Run your existing eval scripts against these two new directories.
The comparison you now have:

| Model | arXiv-only | 50/50 mixed |
|---|---|---|
| all-MiniLM-L6-v2 (22M) | ✅ existing | ✅ existing |
| bge-base-en-v1.5 (109M) | ✅ this run | ✅ this run |
| nomic-embed-text (baseline, no FT) | ✅ existing | — |

If bge-base fine-tuned also underperforms nomic baseline → the gap is **fine-tuning scale**, not model size.
If bge-base fine-tuned matches or beats nomic → the gap was model size all along.